# 02 — Preprocessing, Sentinel Elimination & Chronological Embargo Partition
**Analytics and Systems of Big Data Practice — Endsem Hackathon**
*IIITDM Kancheepuram*

## 1. Sentinel Detection and Elimination
Non-NaN sentinels: 0.0 in US rate columns (15 dates) and -1.0 in foreign returns (40 dates).

In [1]:
import sys, os
from pathlib import Path
ROOT = Path(os.getcwd()).resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import pandas as pd, numpy as np
from src.config import *
from IPython.display import Image, display

from src.preprocessing import clean_sentinels, truncate_warmup, execute_preprocessing_pipeline
processed_dfs = execute_preprocessing_pipeline()
df_sentinel_audit = pd.read_csv(TABLES_DIR / 'table_02_sentinel_audit.csv')
print('Total Sentinel Instances Found:', len(df_sentinel_audit))
df_sentinel_audit.head(10)

Total Sentinel Instances Found: 1740


,Column,Sentinel_Type,Sentinel_Value,Date,Index
0,DGS10,Zero_Rate,0.0,2010-10-11,DJI
1,DGS10,Zero_Rate,0.0,2010-11-11,DJI
2,DGS10,Zero_Rate,0.0,2011-10-10,DJI
3,DGS10,Zero_Rate,0.0,2011-11-11,DJI
4,DGS10,Zero_Rate,0.0,2012-10-08,DJI
5,DGS10,Zero_Rate,0.0,2012-11-12,DJI
6,DGS10,Zero_Rate,0.0,2013-10-14,DJI
7,DGS10,Zero_Rate,0.0,2013-11-11,DJI
8,DGS10,Zero_Rate,0.0,2014-10-13,DJI
9,DGS10,Zero_Rate,0.0,2014-11-11,DJI


## 2. Warm-Up Truncation & Representations A and B
Remove the first 200 warm-up rows (leaving 1,784 clean rows).

In [2]:
df_trunc = pd.read_csv(TABLES_DIR / 'table_03_truncation_summary.csv')
df_trunc

,Index,Raw_Rows,Warmup_Truncated,Rep_A_Rows,Rep_A_NaNs,Rep_B_Rows,Rep_B_NaNs,Rep_A_Start_Date,Rep_A_End_Date
0,DJI,1984,200,1784,2701,1784,0,2010-10-18,2017-11-15
1,NASDAQ,1984,200,1784,2701,1784,0,2010-10-18,2017-11-15
2,NYSE,1984,200,1784,2701,1784,0,2010-10-18,2017-11-15
3,RUSSELL,1984,200,1784,2701,1784,0,2010-10-18,2017-11-15
4,S&P,1984,200,1784,2701,1784,0,2010-10-18,2017-11-15


## 3. Chronological Train-Embargo-Test Partition
Partition: 1,425 Train + 1 Embargo + 357 Test = 1,783 Labeled observations.

In [3]:
df_splits = pd.read_csv(TABLES_DIR / 'table_04_split_partitions.csv')
df_splits

,Index,Train_Rows,Train_Start,Train_End,Embargo_Rows,Embargo_Date,Test_Rows,Test_Start,Test_End
0,DJI,1425,2010-10-18,2016-06-15,1,2016-06-16,357,2016-06-17,2017-11-14
1,NASDAQ,1425,2010-10-18,2016-06-15,1,2016-06-16,357,2016-06-17,2017-11-14
2,NYSE,1425,2010-10-18,2016-06-15,1,2016-06-16,357,2016-06-17,2017-11-14
3,RUSSELL,1425,2010-10-18,2016-06-15,1,2016-06-16,357,2016-06-17,2017-11-14
4,S&P,1425,2010-10-18,2016-06-15,1,2016-06-16,357,2016-06-17,2017-11-14


## 4. Test Target Distribution and Honest Baseline
Verify the exact test majority class ratio (55.46%).

In [4]:
df_target_dist = pd.read_csv(TABLES_DIR / 'table_05_target_distribution_S&P.csv')
df_target_dist

,Index,Total_Labeled_Rows,Test_Rows,Test_NextDay_Up,Test_NextDay_NotUp,Test_Up_Ratio,Expected_Majority_Baseline
0,S&P,1783,357,198,159,0.554622,55.46%
